<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-01-logica/05_Formas_Normais_e_Otimizacao_Booleana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 05 - Notebook: Formas Normais (FND/FNC) e Otimizador Booleano

Neste notebook implementamos algoritmos para geração de **Forma Normal Disjuntiva (FND)**, **Forma Normal Conjuntiva (FNC)** e extração de mintermos/maxtermos aplicados à lógica de acionamento do Drone Agrícola.

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

import itertools
from typing import List, Dict, Callable, Tuple

class OtimizadorBooleano:
    @staticmethod
    def extrair_mintermos_maxtermos(variaveis: List[str], fn_alvo: Callable[[Dict[str, bool]], bool]) -> Tuple[List[Dict[str, bool]], List[Dict[str, bool]]]:
        mintermos = []
        maxtermos = []
        for combo in itertools.product([False, True], repeat=len(variaveis)):
            env = dict(zip(variaveis, combo))
            if fn_alvo(env):
                mintermos.append(env)
            else:
                maxtermos.append(env)
        return mintermos, maxtermos

    @staticmethod
    def formatar_fnd_canonico(variaveis: List[str], mintermos: List[Dict[str, bool]]) -> str:
        termos = []
        for m in mintermos:
            partes = [v if m[v] else f"not_{v}" for v in variaveis]
            termos.append("(" + " AND ".join(partes) + ")")
        return " OR ".join(termos) if termos else "FALSO"

    @staticmethod
    def formatar_fnc_canonico(variaveis: List[str], maxtermos: List[Dict[str, bool]]) -> str:
        termos = []
        for M in maxtermos:
            partes = [f"not_{v}" if M[v] else v for v in variaveis]
            termos.append("(" + " OR ".join(partes) + ")")
        return " AND ".join(termos) if termos else "VERDADEIRO"

# ==========================================
# Aplicação: Drone Agrícola (Bomba de Calda)
# ==========================================
def permissivo_drone_simplificado(env: Dict[str, bool]) -> bool:
    """
    A bomba de pulverização só liga se:
    Voando (is_flying) E Altitude Correta (alt_ok) E Tanque NÃO Vazio (not tank_empty)
    """
    return env['is_flying'] and env['alt_ok'] and (not env['tank_empty'])

# Definindo as variáveis de telemetria
variaveis = ['is_flying', 'alt_ok', 'tank_empty']
mintermos, maxtermos = OtimizadorBooleano.extrair_mintermos_maxtermos(variaveis, permissivo_drone_simplificado)

print("--- FORMA NORMAL DISJUNTIVA CANÔNICA (FND) ---")
print("Lógica para ACIONAR a bomba:")
print(OtimizadorBooleano.formatar_fnd_canonico(variaveis, mintermos))

print("\n--- FORMA NORMAL CONJUNTIVA CANÔNICA (FNC) ---")
print("Lógica para BLOQUEAR a bomba (todas as condições de falha mapeadas):")
print(OtimizadorBooleano.formatar_fnc_canonico(variaveis, maxtermos))

--- FORMA NORMAL DISJUNTIVA CANÔNICA (FND) ---
Lógica para ACIONAR a bomba:
(is_flying AND alt_ok AND not_tank_empty)

--- FORMA NORMAL CONJUNTIVA CANÔNICA (FNC) ---
Lógica para BLOQUEAR a bomba (todas as condições de falha mapeadas):
(is_flying OR alt_ok OR tank_empty) AND (is_flying OR alt_ok OR not_tank_empty) AND (is_flying OR not_alt_ok OR tank_empty) AND (is_flying OR not_alt_ok OR not_tank_empty) AND (not_is_flying OR alt_ok OR tank_empty) AND (not_is_flying OR alt_ok OR not_tank_empty) AND (not_is_flying OR not_alt_ok OR not_tank_empty)
